# Nocturne Aegis Cel Candidate Generator

Generate **80 candidate images** for a future `nacel_v1` LoRA:

- **10 prompts**
- **8 images per prompt**
- base model: `OnomaAIResearch/Illustrious-xl-early-release-v0`
- outputs: PNG images, `.txt` LoRA captions, metadata CSV/JSONL, contact sheet, ZIP archive

Use this notebook to generate candidates, then manually select the best 24–30 clean images for LoRA training.

In [ ]:
# Colab setup
!pip install -q -U diffusers transformers accelerate safetensors huggingface_hub pillow pandas

In [ ]:
# Optional: mount Google Drive so outputs persist after the Colab runtime disconnects.
# Uncomment these lines if you want to save directly to Drive.

# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Optional: Hugging Face login.
# Use this only if the model requires accepting a license or you hit access/rate limits.

# from huggingface_hub import login
# login()

In [ ]:
import os
import json
import random
import gc
from pathlib import Path
from datetime import datetime

import torch
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
from diffusers import DiffusionPipeline, EulerAncestralDiscreteScheduler

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))

## Configuration

Edit these values first. For Colab T4, keep `USE_CPU_OFFLOAD = True` if you run out of VRAM. For A100/L4, `False` is usually faster.

In [ ]:
MODEL_ID = "OnomaAIResearch/Illustrious-xl-early-release-v0"
PROJECT_NAME = "nocturne_aegis_candidates"

# For Drive persistence, change this to something like:
# OUTPUT_ROOT = "/content/drive/MyDrive/nocturne_aegis_outputs"
OUTPUT_ROOT = "/content/outputs"

NUM_IMAGES_PER_PROMPT = 8
BASE_SEED = 24052026

# Good starting points:
# square: 1024 x 1024
# portrait/full body: 832 x 1216
# wide environment: 1216 x 832
WIDTH = 1024
HEIGHT = 1024

NUM_INFERENCE_STEPS = 32
GUIDANCE_SCALE = 6.0

# If True, saves memory but is slower. Useful on T4.
USE_CPU_OFFLOAD = True

# If True, uses a scheduler that usually works well for anime illustration candidates.
USE_EULER_A = True

output_dir = Path(OUTPUT_ROOT) / PROJECT_NAME
images_dir = output_dir / "images"
images_dir.mkdir(parents=True, exist_ok=True)

print("Output directory:", output_dir)

## Style Core

Keep this stable across candidate generation. Only vary the content prompts.

In [ ]:
STYLE_CORE = """
Rendered in Nocturne Aegis Cel: original high-end digital anime illustration with a 1990s cel-animation skeleton and modern volumetric digital painting finish. Elegant 8-head figure ratio where applicable, elongated tapering proportions, fragile anatomical tension, sharp geometric facial planes, angular jaw, subtle melancholic micro-expression.

Delicate sweeping linework with tapered edges for hair, cloth, skin, and face; bold structural ink contours for armor, weapons, machinery, and architecture; selective cross-hatching only in deep shadow recesses. Hair, fabric, cables, smoke, or energy trails form compositional arcs and intentional negative-space framing.

Saturated jewel-toned palette with subtle digital gradient maps: blackened indigo, violet, magenta, sapphire, antique gold, pale silver highlights. High-contrast flat color blocking beneath layered volumetric shading.

Layered 1/2/3 anime shadow system: hard cel-shadow terminators, secondary form shadows, deep ambient occlusion in seams and crevices. Extreme chiaroscuro, strong directional key light, controlled fill, high-intensity rim light separating the silhouette.

Glass-like anime eyes with geometric iris structure, jewel gradients, crisp upper-lid shadow, large catchlights. Skin has soft diffuse glow; cloth has matte velvet texture; metal has iridescent reflections, sharp highlights, burnished chrome edges, black ink recesses, fine micro-scratches.

Cinematic isolation, ethereal fantasy atmosphere, psychological tension, controlled ornamental density, crisp facial read, clean hands, clear silhouette, intentional detail hierarchy, no text, no watermark.
""".strip()

NEGATIVE_PROMPT = """
generic anime, generic modern anime, bland moe face, chibi, round childish proportions, plastic skin, doll-like face, photorealistic skin texture, 3d render, western comic style, painterly mush, low contrast, flat lighting, muddy colors, uncontrolled neon bloom, blurry lineart, uniform line weight, messy ornamental noise, over-detailed background competing with face, bad anatomy, malformed hands, extra fingers, missing fingers, broken wrists, duplicated limbs, fused armor plates, unreadable weapon, cropped feet, cropped hands, text, logo, watermark, signature, jpeg artifacts
""".strip()

print("Style core length:", len(STYLE_CORE))
print("Negative prompt length:", len(NEGATIVE_PROMPT))

## Prompts

Edit these 10 entries. Each entry has:

- `id`: short unique filename prefix
- `content_prompt`: what to generate; the style core is appended automatically
- `caption`: content-only LoRA caption beginning with `nacel_v1`

Do **not** put style terms in captions. Captions should describe content only.

In [ ]:
PROMPTS = [
    {
        "id": "nacel_001",
        "content_prompt": "An adult sky-pilot standing on a broken launch bridge, one hand resting on a cracked helmet, wind pulling long coat panels into sweeping arcs, distant gothic starships embedded in a ruined skyline, low-angle full-body composition, moonlit backlight",
        "caption": "nacel_v1, an adult sky-pilot standing on a broken launch bridge with one hand resting on a cracked helmet, long coat panels blowing in the wind, distant starships in a ruined skyline, low-angle full-body composition"
    },
    {
        "id": "nacel_002",
        "content_prompt": "An adult male swordsman with short silver hair seated inside a damaged cockpit, armored coat open at the collar, one hand holding a sheathed blade, three-quarter bust composition, instrument lights glowing behind him",
        "caption": "nacel_v1, an adult man with short silver hair seated inside a damaged cockpit, wearing an armored coat and holding a sheathed blade, three-quarter bust composition"
    },
    {
        "id": "nacel_003",
        "content_prompt": "A masked androgynous knight standing in a flooded cathedral corridor, reflective black water at their feet, narrow vertical windows behind them, long ribbons and cables forming arcs around the body, centered full-body composition",
        "caption": "nacel_v1, a masked androgynous knight standing in a flooded cathedral corridor with reflective water, long ribbons and cables around the body, centered full-body composition"
    },
    {
        "id": "nacel_004",
        "content_prompt": "An adult woman with cropped dark hair repairing a biomechanical gauntlet on a workbench, tools and small metal parts arranged around her, focused downward expression, close portrait with hands visible, dark workshop background",
        "caption": "nacel_v1, an adult woman with cropped dark hair repairing a biomechanical gauntlet on a workbench, tools and metal parts around her, close portrait with hands visible"
    },
    {
        "id": "nacel_005",
        "content_prompt": "A ceremonial armored priest standing before a suspended moon engine, layered black robes with pale metal shoulder guards, one hand raised in a restrained ritual gesture, wide environmental shot with monumental vertical architecture",
        "caption": "nacel_v1, a ceremonial armored priest standing before a suspended moon engine, wearing layered black robes and pale metal shoulder guards, one hand raised, wide environmental shot"
    },
    {
        "id": "nacel_006",
        "content_prompt": "A winged humanoid machine kneeling on a cathedral bridge, folded mechanical wings, visible joints and plated limbs, no human rider, distant city ruins below, dramatic wide shot from a high angle",
        "caption": "nacel_v1, a winged humanoid machine kneeling on a cathedral bridge with folded mechanical wings and visible joints, distant city ruins below, high-angle wide shot"
    },
    {
        "id": "nacel_007",
        "content_prompt": "An adult archer in layered traveling clothes and light armor standing in a ruined observatory, bow lowered at their side, star maps and broken brass instruments in the background, calm profile full-body pose",
        "caption": "nacel_v1, an adult archer wearing layered traveling clothes and light armor standing in a ruined observatory, holding a lowered bow, calm profile full-body pose"
    },
    {
        "id": "nacel_008",
        "content_prompt": "An ornate helmet with a glass visor resting on dark velvet fabric, antique metal trim, fine scratches, broken feather-like crest, object study composition, shallow background with no person visible",
        "caption": "nacel_v1, an ornate helmet with a glass visor resting on dark velvet fabric, antique metal trim, fine scratches, broken crest, object study composition"
    },
    {
        "id": "nacel_009",
        "content_prompt": "An armored celestial wolf standing on a cracked stone platform, mechanical halo fragments floating behind its head, long fur and metal plates interwoven, full-body creature study, storm clouds and moonlight in the background",
        "caption": "nacel_v1, an armored celestial wolf standing on a cracked stone platform with mechanical halo fragments behind its head, long fur and metal plates interwoven, full-body creature study"
    },
    {
        "id": "nacel_010",
        "content_prompt": "A gothic starfighter parked on a rain-slick launch rail, exposed wing mechanisms, narrow cockpit canopy, trailing cables and banners around the landing gear, low-angle vehicle study with distant ruined towers",
        "caption": "nacel_v1, a gothic starfighter parked on a rain-slick launch rail with exposed wing mechanisms, narrow cockpit canopy, cables and banners around the landing gear, low-angle vehicle study"
    }
]

assert len(PROMPTS) == 10, f"Expected 10 prompts, got {len(PROMPTS)}"
for item in PROMPTS:
    assert item["caption"].startswith("nacel_v1,"), f"Caption must start with nacel_v1: {item['id']}"

pd.DataFrame(PROMPTS)

In [ ]:
# Optional: load prompts from an uploaded prompts.json file instead of the cell above.
# In Colab, upload prompts.json to /content/prompts.json, then run this cell.

prompts_json_path = Path("/content/prompts.json")
if prompts_json_path.exists():
    PROMPTS = json.loads(prompts_json_path.read_text())
    print(f"Loaded {len(PROMPTS)} prompts from {prompts_json_path}")
else:
    print("Using prompts from notebook cell.")

## Load model

The notebook uses `float16` on most Colab GPUs and `bfloat16` on Ampere-or-newer GPUs when available.

In [ ]:
def pick_dtype():
    if not torch.cuda.is_available():
        return torch.float32
    major, minor = torch.cuda.get_device_capability(0)
    # Ampere and newer generally support bfloat16. T4 does not, so it uses float16.
    return torch.bfloat16 if major >= 8 else torch.float16

DTYPE = pick_dtype()
print("Using dtype:", DTYPE)

pipe = DiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    use_safetensors=True,
)

if USE_EULER_A:
    pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)

pipe.enable_vae_slicing()
try:
    pipe.enable_vae_tiling()
except Exception:
    pass

if torch.cuda.is_available():
    if USE_CPU_OFFLOAD:
        pipe.enable_model_cpu_offload()
        print("Model CPU offload enabled.")
    else:
        pipe.to("cuda")
        print("Pipeline moved to CUDA.")
else:
    print("CUDA unavailable. This will be very slow on CPU.")

## Generate 80 candidates

This saves:

- image PNGs
- sidecar `.txt` captions for LoRA training
- `metadata.csv`
- `metadata.jsonl`

In [ ]:
def build_full_prompt(content_prompt: str) -> str:
    return f"{content_prompt}.\n\n{STYLE_CORE}"

metadata_rows = []
start_time = datetime.utcnow().isoformat()

for prompt_index, item in enumerate(PROMPTS, start=1):
    prompt_id = item["id"]
    content_prompt = item["content_prompt"].strip()
    caption = item["caption"].strip()
    full_prompt = build_full_prompt(content_prompt)

    print(f"\n=== {prompt_index:02d}/{len(PROMPTS)} | {prompt_id} ===")

    for variation_index in range(1, NUM_IMAGES_PER_PROMPT + 1):
        seed = BASE_SEED + (prompt_index * 1000) + variation_index
        generator_device = "cuda" if torch.cuda.is_available() else "cpu"
        generator = torch.Generator(device=generator_device).manual_seed(seed)

        filename_base = f"{prompt_id}_seed_{seed}_{variation_index:02d}"
        image_path = images_dir / f"{filename_base}.png"
        caption_path = images_dir / f"{filename_base}.txt"

        if image_path.exists():
            print(f"Skipping existing: {image_path.name}")
            continue

        with torch.inference_mode():
            result = pipe(
                prompt=full_prompt,
                negative_prompt=NEGATIVE_PROMPT,
                width=WIDTH,
                height=HEIGHT,
                num_inference_steps=NUM_INFERENCE_STEPS,
                guidance_scale=GUIDANCE_SCALE,
                generator=generator,
            )

        image = result.images[0]
        image.save(image_path)
        caption_path.write_text(caption + "\n", encoding="utf-8")

        metadata_rows.append({
            "filename": image_path.name,
            "caption_filename": caption_path.name,
            "prompt_id": prompt_id,
            "prompt_index": prompt_index,
            "variation_index": variation_index,
            "seed": seed,
            "model_id": MODEL_ID,
            "width": WIDTH,
            "height": HEIGHT,
            "num_inference_steps": NUM_INFERENCE_STEPS,
            "guidance_scale": GUIDANCE_SCALE,
            "content_prompt": content_prompt,
            "caption": caption,
            "negative_prompt": NEGATIVE_PROMPT,
            "generated_at_utc": datetime.utcnow().isoformat(),
        })

        print(f"Saved: {image_path.name}")

        # Keep Colab memory stable.
        del result, image
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

metadata_df = pd.DataFrame(metadata_rows)

# If this run skipped existing files, rebuild metadata from this run only may be partial. It still records newly generated files.
metadata_csv_path = output_dir / "metadata.csv"
metadata_jsonl_path = output_dir / "metadata.jsonl"
metadata_df.to_csv(metadata_csv_path, index=False)
metadata_df.to_json(metadata_jsonl_path, orient="records", lines=True, force_ascii=False)

print("\nDone.")
print("Generated this run:", len(metadata_df))
print("Metadata:", metadata_csv_path)

## Make contact sheet

Use this to quickly reject broken generations before LoRA training.

In [ ]:
def make_contact_sheet(image_paths, output_path, thumb_size=(256, 256), cols=8):
    image_paths = list(image_paths)
    if not image_paths:
        print("No images found.")
        return None

    rows = (len(image_paths) + cols - 1) // cols
    label_h = 34
    sheet = Image.new("RGB", (cols * thumb_size[0], rows * (thumb_size[1] + label_h)), "white")
    draw = ImageDraw.Draw(sheet)

    for idx, path in enumerate(image_paths):
        img = Image.open(path).convert("RGB")
        img.thumbnail(thumb_size, Image.LANCZOS)
        x = (idx % cols) * thumb_size[0]
        y = (idx // cols) * (thumb_size[1] + label_h)
        x_img = x + (thumb_size[0] - img.width) // 2
        y_img = y
        sheet.paste(img, (x_img, y_img))
        draw.text((x + 4, y + thumb_size[1] + 4), path.stem[:34], fill=(0, 0, 0))

    sheet.save(output_path, quality=92)
    return sheet

all_images = sorted(images_dir.glob("*.png"))
contact_sheet_path = output_dir / "contact_sheet.jpg"
sheet = make_contact_sheet(all_images, contact_sheet_path, cols=8)
print("Contact sheet:", contact_sheet_path)
sheet

## Create ZIP archive

Download this archive from the Colab file browser, or copy it to Drive.

In [ ]:
import shutil

archive_base = str(output_dir / "output_archive")
zip_path = shutil.make_archive(archive_base, "zip", root_dir=output_dir)
print("ZIP archive created:", zip_path)

# Optional Colab download:
# from google.colab import files
# files.download(zip_path)

## Candidate audit checklist

For the final LoRA dataset, keep only images that pass most of these checks:

- clear face/eyes or intentionally no face for object/vehicle studies
- clean hands when hands are visible
- readable silhouette
- no text/logo/watermark/signature
- no melted armor or fused objects
- strong linework hierarchy
- good material separation: skin, cloth, metal, glass, energy
- not too similar to the purple armored sword-girl seed image
- good content diversity across humans, objects, creatures, vehicles, environments

Recommended final dataset: **24–30 images** from the 80 generated candidates.